# PTCG Strategy MRI: From 2,022 Cards to 60 Decisions

> **Thesis.** A strong deck is not a ranking of individually strong cards. It
> is a constrained control system whose value comes from legal, reachable
> action sequences under hidden information.

This notebook is an independently implemented strategy laboratory for the
[Pokémon TCG AI Battle Challenge Strategy](https://www.kaggle.com/competitions/pokemon-tcg-ai-battle-challenge-strategy).
It connects the official card catalogue to bounded text parsing, multi-objective
efficiency, an auditable 60-card integer portfolio, exact opening consistency,
and a fail-closed policy order.

## Evidence labels used throughout

| Label | Meaning |
|---|---|
| **Catalogue aggregate** | Describes only the supplied competition data. |
| **Synthetic fixture** | Tests mechanics; it is not evidence of game strength. |
| **Local paired match** | Local evidence; never a Kaggle score. |
| **Official COMPLETE row** | Time-specific rating only when the exact row has a nonempty score. |
| **Strategy writeup** | Hackathon artifact, judged separately from simulation rating. |
| **Notebook votes** | Public Code surface; raw votes are not medal-eligibility proof. |

The notebook creates no competition submission. It also does not redistribute
the source CSV as an output.


## Given — contract before charts

The catalogue contains one row per move for many Pokémon, so row count is not
card count. Missing values are also structural: Energy cards do not have HP,
Trainer cards do not have retreat costs, and some attacks have conditional
damage that must not be coerced into a fake number.

We first verify the exact 17-column interface, bind the observed file hash, and
collapse move rows to one auditable record per Card ID.


In [ ]:
"""Clean-room mechanics for the PTCG Strategy Decision Atlas.

This module deliberately implements only bounded catalogue parsing, transparent
portfolio constraints, and statistical diagnostics.  It does not reproduce a
public competitor's deck engine or agent policy.
"""

from __future__ import annotations

import hashlib
import math
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import csr_matrix


EXPECTED_COLUMNS = (
    "Card ID",
    "Card Name",
    "Expansion",
    "Collection No.",
    "Stage (Pokémon)/Type (Energy and Trainer)",
    "Rule",
    "Category",
    "Previous stage",
    "HP",
    "Type",
    "Weakness",
    "Resistance (Type)",
    "Retreat",
    "Move Name",
    "Cost",
    "Damage",
    "Effect Explanation",
)

KNOWN_EN_SHA256 = "a0ea63cf7adcb65d35436ce0eb390de6e2e35654a7c67c065a45f4abaa00f373"


def sha256_file(path: str | Path) -> str:
    """Return the SHA256 digest of a file without loading it twice."""

    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_catalogue_path(candidates: Sequence[str | Path] | None = None) -> Path:
    """Resolve the official English catalogue from explicit, bounded locations."""

    if candidates is None:
        kaggle_input_root = Path("/kaggle/input")
        discovered = (
            tuple(sorted(kaggle_input_root.rglob("EN_Card_Data.csv")))
            if kaggle_input_root.is_dir()
            else ()
        )
        candidates = (
            Path("/kaggle/input/pokemon-tcg-ai-battle-challenge-strategy/EN_Card_Data.csv"),
            Path("/kaggle/input/competitions/pokemon-tcg-ai-battle-challenge-strategy/EN_Card_Data.csv"),
            Path("../input/pokemon-tcg-ai-battle-challenge-strategy/EN_Card_Data.csv"),
            Path("data/EN_Card_Data.csv"),
            Path("research/pokemon_strategy_decision_atlas_2026_07_28/data/EN_Card_Data.csv"),
            *discovered,
        )
    existing_by_resolved_path = {
        str(Path(candidate).resolve()): Path(candidate)
        for candidate in candidates
        if Path(candidate).is_file()
    }
    existing = list(existing_by_resolved_path.values())
    if len(existing) != 1:
        raise FileNotFoundError(
            "Expected exactly one EN_Card_Data.csv in the bounded search paths; "
            f"found {len(existing)}: {[str(path) for path in existing]}"
        )
    return existing[0]


def validate_catalogue(df: pd.DataFrame) -> dict[str, int]:
    """Fail closed on the documented schema and basic identifier invariants."""

    columns = tuple(df.columns)
    if columns != EXPECTED_COLUMNS:
        raise ValueError(f"Unexpected catalogue columns: {columns}")
    if df.empty:
        raise ValueError("Catalogue is empty")
    if df["Card ID"].isna().any():
        raise ValueError("Card ID contains missing values")
    ids = pd.to_numeric(df["Card ID"], errors="raise")
    if (ids <= 0).any():
        raise ValueError("Card IDs must be positive")
    if df["Card Name"].isna().any():
        raise ValueError("Card Name contains missing values")
    return {
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "unique_card_ids": int(ids.nunique()),
        "move_rows_beyond_unique_cards": int(len(df) - ids.nunique()),
    }


def _clean_text(value: object) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def parse_energy_cost(value: object) -> dict[str, object]:
    """Parse visible energy symbols without interpreting effect prose."""

    text = _clean_text(value)
    tokens = re.findall(r"\{[^{}]+\}|●", text)
    typed = tuple(token for token in tokens if token != "●")
    return {
        "cost_text": text,
        "energy_total": len(tokens),
        "energy_typed": len(typed),
        "energy_colorless": tokens.count("●"),
        "energy_types": typed,
        "fully_parsed": bool(text) and "".join(tokens) == re.sub(r"\s+", "", text),
    }


def parse_damage(value: object) -> dict[str, object]:
    """Parse only unconditional integer damage and label every other form."""

    text = _clean_text(value)
    if not text:
        return {"damage_text": "", "damage_value": np.nan, "damage_kind": "missing"}
    if re.fullmatch(r"\d+", text):
        return {
            "damage_text": text,
            "damage_value": float(int(text)),
            "damage_kind": "deterministic",
        }
    if "×" in text or "x" in text.lower():
        kind = "multiplier"
    elif "+" in text:
        kind = "conditional_addition"
    elif "-" in text:
        kind = "modifier_or_reduction"
    else:
        kind = "unparsed"
    return {"damage_text": text, "damage_value": np.nan, "damage_kind": kind}


def classify_card(stage_or_type: object) -> str:
    text = _clean_text(stage_or_type)
    if text in {"Item", "Supporter", "Pokémon Tool", "Stadium"}:
        return "Trainer"
    if text in {"Basic Pokémon", "Stage 1 Pokémon", "Stage 2 Pokémon"}:
        return "Pokemon"
    if "Energy" in text:
        return "Energy"
    return "Other"


def _first_nonempty(values: Iterable[object]) -> object:
    for value in values:
        if value is not None and not pd.isna(value) and str(value).strip():
            return value
    return np.nan


def _effect_keywords(text: str) -> dict[str, int]:
    lowered = text.lower()
    groups = {
        "draw": ("draw",),
        "search": ("search", "look at"),
        "switch": ("switch", "retreat"),
        "energy": ("energy", "attach"),
        "heal": ("heal", "recover"),
        "disrupt": ("discard", "shuffle", "opponent"),
    }
    return {name: int(any(token in lowered for token in tokens)) for name, tokens in groups.items()}


def build_card_table(raw: pd.DataFrame) -> pd.DataFrame:
    """Collapse move rows to one auditable record per Card ID."""

    validate_catalogue(raw)
    working = raw.copy()
    cost_rows = working["Cost"].map(parse_energy_cost).apply(pd.Series)
    damage_rows = working["Damage"].map(parse_damage).apply(pd.Series)
    working = pd.concat([working, cost_rows, damage_rows], axis=1)
    records: list[dict[str, object]] = []

    for card_id, group in working.groupby("Card ID", sort=True, dropna=False):
        deterministic = group[group["damage_kind"] == "deterministic"].copy()
        positive_cost = deterministic[deterministic["energy_total"] > 0].copy()
        if positive_cost.empty:
            best_dpe = np.nan
        else:
            best_dpe = float((positive_cost["damage_value"] / positive_cost["energy_total"]).max())
        effect_parts = []
        for value in group["Effect Explanation"]:
            text = _clean_text(value)
            if text and text not in effect_parts:
                effect_parts.append(text)
        effect_text = " | ".join(effect_parts)
        stage = _clean_text(_first_nonempty(group["Stage (Pokémon)/Type (Energy and Trainer)"]))
        kind = classify_card(stage)
        damage_nonmissing = group["damage_kind"] != "missing"
        record = {
            "Card ID": int(card_id),
            "Card Name": _clean_text(_first_nonempty(group["Card Name"])),
            "Expansion": _clean_text(_first_nonempty(group["Expansion"])),
            "StageOrTrainerType": stage,
            "Rule": _clean_text(_first_nonempty(group["Rule"])),
            "PreviousStage": _clean_text(_first_nonempty(group["Previous stage"])),
            "HP": float(pd.to_numeric(group["HP"], errors="coerce").max()),
            "Type": _clean_text(_first_nonempty(group["Type"])),
            "Weakness": _clean_text(_first_nonempty(group["Weakness"])),
            "Resistance": _clean_text(_first_nonempty(group["Resistance (Type)"])),
            "Retreat": float(pd.to_numeric(group["Retreat"], errors="coerce").max()),
            "Kind": kind,
            "MoveRows": int(group["Move Name"].notna().sum()),
            "DeterministicMoveRows": int((group["damage_kind"] == "deterministic").sum()),
            "ConditionalMoveRows": int((damage_nonmissing & (group["damage_kind"] != "deterministic")).sum()),
            "MaxDeterministicDamage": float(deterministic["damage_value"].max())
            if not deterministic.empty
            else np.nan,
            "BestDamagePerEnergy": best_dpe,
            "MinParsedEnergyCost": float(pd.to_numeric(group["energy_total"], errors="coerce").replace(0, np.nan).min()),
            "EffectText": effect_text,
            "EffectLength": len(effect_text),
        }
        record.update(_effect_keywords(effect_text))
        records.append(record)

    cards = pd.DataFrame.from_records(records).sort_values("Card ID").reset_index(drop=True)
    cards["IsBasicPokemon"] = cards["StageOrTrainerType"].eq("Basic Pokémon")
    cards["IsBasicEnergy"] = cards["StageOrTrainerType"].eq("Basic Energy")
    cards["IsAceSpec"] = cards["Rule"].eq("ACE SPEC")
    cards["EvolutionDepth"] = cards["StageOrTrainerType"].map(
        {"Basic Pokémon": 0, "Stage 1 Pokémon": 1, "Stage 2 Pokémon": 2}
    )
    if cards["Card ID"].duplicated().any():
        raise AssertionError("Card-level table still contains duplicate Card IDs")
    return cards


def robust_z(values: pd.Series, *, fill: float = 0.0) -> pd.Series:
    numeric = pd.to_numeric(values, errors="coerce")
    median = float(numeric.median()) if numeric.notna().any() else 0.0
    mad = float((numeric - median).abs().median()) if numeric.notna().any() else 0.0
    scale = 1.4826 * mad
    if not math.isfinite(scale) or scale < 1e-12:
        scale = float(numeric.std(ddof=0)) if numeric.notna().any() else 1.0
    if not math.isfinite(scale) or scale < 1e-12:
        scale = 1.0
    return ((numeric.fillna(median) - median) / scale).clip(-3.0, 3.0).fillna(fill)


def score_cards(cards: pd.DataFrame) -> pd.DataFrame:
    """Attach transparent, bounded utility components by card role."""

    scored = cards.copy()
    scored["durability_z"] = robust_z(scored["HP"])
    scored["pressure_z"] = robust_z(scored["BestDamagePerEnergy"])
    scored["mobility_z"] = -robust_z(scored["Retreat"])
    move_denominator = scored["MoveRows"].replace(0, np.nan)
    scored["conditional_share"] = (scored["ConditionalMoveRows"] / move_denominator).fillna(0.0)
    scored["trainer_signal"] = (
        1.00 * scored["draw"]
        + 1.15 * scored["search"]
        + 0.65 * scored["switch"]
        + 0.75 * scored["energy"]
        + 0.35 * scored["heal"]
        + 0.45 * scored["disrupt"]
    )
    pokemon_utility = (
        0.34 * scored["durability_z"]
        + 0.42 * scored["pressure_z"]
        + 0.18 * scored["mobility_z"]
        - 0.20 * scored["conditional_share"]
        - 0.10 * scored["EvolutionDepth"].fillna(0.0)
    )
    trainer_utility = robust_z(scored["trainer_signal"]) - 0.05 * robust_z(scored["EffectLength"])
    energy_utility = pd.Series(0.0, index=scored.index)
    scored["Utility"] = np.select(
        [scored["Kind"].eq("Pokemon"), scored["Kind"].eq("Trainer"), scored["Kind"].eq("Energy")],
        [pokemon_utility, trainer_utility, energy_utility],
        default=-3.0,
    )
    return scored


def pareto_frontier_mask(values: np.ndarray, maximize: Sequence[bool]) -> np.ndarray:
    """Return nondominated rows for a small, finite multi-objective table."""

    matrix = np.asarray(values, dtype=float)
    if matrix.ndim != 2 or matrix.shape[1] != len(maximize):
        raise ValueError("values/maximize shape mismatch")
    oriented = matrix.copy()
    for column, should_maximize in enumerate(maximize):
        if not should_maximize:
            oriented[:, column] *= -1.0
    keep = np.ones(len(oriented), dtype=bool)
    for idx, point in enumerate(oriented):
        if not np.isfinite(point).all():
            keep[idx] = False
            continue
        dominated = np.all(oriented >= point, axis=1) & np.any(oriented > point, axis=1)
        dominated[idx] = False
        if dominated.any():
            keep[idx] = False
    return keep


def probability_at_least_one(success_cards: int, *, deck_size: int = 60, draws: int = 7) -> float:
    if not 0 <= success_cards <= deck_size:
        raise ValueError("success_cards must lie in [0, deck_size]")
    if not 0 <= draws <= deck_size:
        raise ValueError("draws must lie in [0, deck_size]")
    if success_cards == 0:
        return 0.0
    if deck_size - success_cards < draws:
        return 1.0
    return 1.0 - math.comb(deck_size - success_cards, draws) / math.comb(deck_size, draws)


def probability_basic_and_energy(
    basic_cards: int,
    energy_cards: int,
    *,
    deck_size: int = 60,
    draws: int = 7,
) -> float:
    """Exact inclusion-exclusion probability for two disjoint card roles."""

    if min(basic_cards, energy_cards) < 0 or basic_cards + energy_cards > deck_size:
        raise ValueError("Basic and Energy counts must be nonnegative and disjoint")
    denominator = math.comb(deck_size, draws)

    def miss(count: int) -> float:
        available = deck_size - count
        return 0.0 if available < draws else math.comb(available, draws) / denominator

    miss_basic = miss(basic_cards)
    miss_energy = miss(energy_cards)
    miss_both = miss(basic_cards + energy_cards)
    return 1.0 - miss_basic - miss_energy + miss_both


def monte_carlo_basic_and_energy(
    basic_cards: int,
    energy_cards: int,
    *,
    deck_size: int = 60,
    draws: int = 7,
    trials: int = 200_000,
    seed: int = 20260728,
) -> float:
    if min(basic_cards, energy_cards) < 0 or basic_cards + energy_cards > deck_size:
        raise ValueError("Basic and Energy counts must be nonnegative and disjoint")
    if trials <= 0:
        raise ValueError("trials must be positive")
    population = np.concatenate(
        [
            np.ones(basic_cards, dtype=np.int8),
            np.full(energy_cards, 2, dtype=np.int8),
            np.zeros(deck_size - basic_cards - energy_cards, dtype=np.int8),
        ]
    )
    rng = np.random.default_rng(seed)
    successes = 0
    batch = 10_000
    for start in range(0, trials, batch):
        size = min(batch, trials - start)
        keys = rng.random((size, deck_size))
        indices = np.argpartition(keys, draws - 1, axis=1)[:, :draws]
        hands = population[indices]
        successes += int(((hands == 1).any(axis=1) & (hands == 2).any(axis=1)).sum())
    return successes / trials


@dataclass(frozen=True)
class DeckQuotas:
    pokemon: int = 18
    trainer: int = 28
    energy: int = 14
    minimum_basic_pokemon: int = 8
    minimum_evolved_pokemon: int = 4
    minimum_primary_type_pokemon: int = 12
    maximum_rule_pokemon: int = 8
    minimum_items: int = 8
    minimum_supporters: int = 6
    minimum_stadiums: int = 1
    minimum_tools: int = 1

    @property
    def total(self) -> int:
        return self.pokemon + self.trainer + self.energy


def _eligible_pool(scored: pd.DataFrame, primary_type: str) -> pd.DataFrame:
    pokemon = scored[
        scored["Kind"].eq("Pokemon") & scored["Type"].isin({primary_type, "{C}"})
    ].copy()
    trainer_source = scored[scored["Kind"].eq("Trainer")].copy()
    trainer_parts = []
    subtype_limits = {"Item": 60, "Supporter": 50, "Pokémon Tool": 20, "Stadium": 20}
    for subtype, limit in subtype_limits.items():
        part = trainer_source[trainer_source["StageOrTrainerType"].eq(subtype)].sort_values(
            ["Utility", "Card ID"], ascending=[False, True]
        )
        trainer_parts.append(part.head(limit))
    trainer = pd.concat(trainer_parts, ignore_index=True)
    energy = scored[
        scored["Kind"].eq("Energy")
        & (scored["IsBasicEnergy"] & scored["Type"].eq(primary_type))
    ].copy()
    pool = pd.concat([pokemon, trainer, energy], ignore_index=True)
    pool = pool.sort_values("Card ID").drop_duplicates("Card ID").reset_index(drop=True)

    available_names = set(pool["Card Name"])
    evolved_without_parent = pool["EvolutionDepth"].fillna(0).gt(0) & ~pool["PreviousStage"].isin(
        available_names
    )
    pool = pool.loc[~evolved_without_parent].reset_index(drop=True)
    if not (pool["Kind"].eq("Energy") & pool["IsBasicEnergy"]).any():
        raise ValueError(f"No basic energy row found for primary type {primary_type}")
    return pool


def optimize_demo_deck(
    scored_cards: pd.DataFrame,
    *,
    primary_type: str = "{F}",
    quotas: DeckQuotas = DeckQuotas(),
) -> pd.DataFrame:
    """Solve a transparent 60-card teaching portfolio with integer constraints."""

    if quotas.total != 60:
        raise ValueError("Teaching deck quotas must sum to exactly 60")
    pool = _eligible_pool(scored_cards, primary_type)
    count = len(pool)
    if count == 0:
        raise ValueError("Eligible card pool is empty")

    objective = -pd.to_numeric(pool["Utility"], errors="coerce").fillna(-3.0).to_numpy(float)
    objective += 1e-9 * pool["Card ID"].to_numpy(float)
    lower = np.zeros(count)
    upper = np.where(pool["IsBasicEnergy"].to_numpy(bool), 60.0, 4.0)
    rows: list[np.ndarray] = []
    lows: list[float] = []
    highs: list[float] = []

    def add_constraint(weights: np.ndarray, low: float, high: float) -> None:
        rows.append(weights.astype(float))
        lows.append(float(low))
        highs.append(float(high))

    add_constraint(np.ones(count), 60, 60)
    for kind, quota in (("Pokemon", quotas.pokemon), ("Trainer", quotas.trainer), ("Energy", quotas.energy)):
        add_constraint(pool["Kind"].eq(kind).to_numpy(float), quota, quota)
    add_constraint(pool["IsBasicPokemon"].to_numpy(float), quotas.minimum_basic_pokemon, np.inf)
    evolved = pool["Kind"].eq("Pokemon") & pool["EvolutionDepth"].fillna(0).gt(0)
    add_constraint(evolved.to_numpy(float), quotas.minimum_evolved_pokemon, np.inf)
    primary_pokemon = pool["Kind"].eq("Pokemon") & pool["Type"].eq(primary_type)
    add_constraint(primary_pokemon.to_numpy(float), quotas.minimum_primary_type_pokemon, np.inf)
    rule_pokemon = pool["Kind"].eq("Pokemon") & pool["Rule"].ne("")
    add_constraint(rule_pokemon.to_numpy(float), -np.inf, quotas.maximum_rule_pokemon)
    for subtype, minimum in (
        ("Item", quotas.minimum_items),
        ("Supporter", quotas.minimum_supporters),
        ("Stadium", quotas.minimum_stadiums),
        ("Pokémon Tool", quotas.minimum_tools),
    ):
        add_constraint(pool["StageOrTrainerType"].eq(subtype).to_numpy(float), minimum, np.inf)
    add_constraint(pool["IsAceSpec"].to_numpy(float), -np.inf, 1)

    for _, indices in pool.groupby("Card Name", sort=True).groups.items():
        idx = np.asarray(list(indices), dtype=int)
        if bool(pool.loc[idx, "IsBasicEnergy"].all()):
            continue
        weights = np.zeros(count)
        weights[idx] = 1.0
        add_constraint(weights, -np.inf, 4)

    evolved_pool = pool.loc[pool["EvolutionDepth"].fillna(0).gt(0) & pool["PreviousStage"].ne("")]
    for parent, child_indices in evolved_pool.groupby("PreviousStage", sort=True).groups.items():
        parent_idx = np.flatnonzero(pool["Card Name"].eq(parent).to_numpy())
        if not len(parent_idx):
            continue
        weights = np.zeros(count)
        weights[np.asarray(list(child_indices), dtype=int)] = 1.0
        weights[parent_idx] -= 1.0
        add_constraint(weights, -np.inf, 0)

    matrix = csr_matrix(np.vstack(rows))
    result = milp(
        c=objective,
        integrality=np.ones(count, dtype=int),
        bounds=Bounds(lower, upper),
        constraints=LinearConstraint(matrix, np.asarray(lows), np.asarray(highs)),
        options={"presolve": True, "time_limit": 30.0},
    )
    if not result.success or result.x is None:
        raise RuntimeError(f"MILP did not find a feasible teaching deck: {result.message}")
    copies = np.rint(result.x).astype(int)
    deck = pool.loc[copies > 0].copy()
    deck["Copies"] = copies[copies > 0]
    deck["PrimaryTypeScenario"] = primary_type
    deck = deck.sort_values(["Kind", "Utility", "Card Name"], ascending=[True, False, True])
    validate_demo_deck(deck, quotas=quotas)
    return deck.reset_index(drop=True)


def validate_demo_deck(deck: pd.DataFrame, *, quotas: DeckQuotas = DeckQuotas()) -> dict[str, object]:
    required = {
        "Card ID",
        "Card Name",
        "Kind",
        "StageOrTrainerType",
        "PreviousStage",
        "IsBasicPokemon",
        "IsBasicEnergy",
        "IsAceSpec",
        "Copies",
    }
    missing = required.difference(deck.columns)
    if missing:
        raise ValueError(f"Deck table is missing columns: {sorted(missing)}")
    copies = pd.to_numeric(deck["Copies"], errors="raise")
    if (copies <= 0).any() or not np.allclose(copies, np.rint(copies)):
        raise ValueError("Deck copies must be positive integers")
    if int(copies.sum()) != quotas.total:
        raise ValueError(f"Deck contains {int(copies.sum())} rather than {quotas.total} cards")
    role_counts = deck.groupby("Kind")["Copies"].sum().to_dict()
    expected = {"Pokemon": quotas.pokemon, "Trainer": quotas.trainer, "Energy": quotas.energy}
    if {key: int(role_counts.get(key, 0)) for key in expected} != expected:
        raise ValueError(f"Role quotas violated: {role_counts}")
    ordinary = deck.loc[~deck["IsBasicEnergy"]]
    name_counts = ordinary.groupby("Card Name")["Copies"].sum()
    if (name_counts > 4).any():
        raise ValueError(f"Ordinary name copy cap violated: {name_counts[name_counts > 4].to_dict()}")
    if int(deck.loc[deck["IsAceSpec"], "Copies"].sum()) > 1:
        raise ValueError("ACE SPEC total exceeds one")
    basic_count = int(deck.loc[deck["IsBasicPokemon"], "Copies"].sum())
    if basic_count < quotas.minimum_basic_pokemon:
        raise ValueError("Minimum Basic Pokemon quota violated")
    evolved_count = int(
        deck.loc[deck["Kind"].eq("Pokemon") & deck["EvolutionDepth"].fillna(0).gt(0), "Copies"].sum()
    )
    if evolved_count < quotas.minimum_evolved_pokemon:
        raise ValueError("Minimum evolved Pokemon scenario constraint violated")
    primary_type = str(deck["PrimaryTypeScenario"].dropna().iloc[0]) if "PrimaryTypeScenario" in deck else ""
    primary_count = int(
        deck.loc[deck["Kind"].eq("Pokemon") & deck["Type"].eq(primary_type), "Copies"].sum()
    )
    if primary_count < quotas.minimum_primary_type_pokemon:
        raise ValueError("Minimum primary-type Pokemon scenario constraint violated")
    rule_count = int(
        deck.loc[deck["Kind"].eq("Pokemon") & deck["Rule"].ne(""), "Copies"].sum()
    )
    if rule_count > quotas.maximum_rule_pokemon:
        raise ValueError("Maximum rule-box Pokemon scenario constraint violated")
    subtype_counts = deck.groupby("StageOrTrainerType")["Copies"].sum().to_dict()
    subtype_minimums = {
        "Item": quotas.minimum_items,
        "Supporter": quotas.minimum_supporters,
        "Stadium": quotas.minimum_stadiums,
        "Pokémon Tool": quotas.minimum_tools,
    }
    for subtype, minimum in subtype_minimums.items():
        if int(subtype_counts.get(subtype, 0)) < minimum:
            raise ValueError(f"Minimum {subtype} scenario constraint violated")
    name_counter = Counter()
    for _, row in deck.iterrows():
        name_counter[str(row["Card Name"])] += int(row["Copies"])
    unsupported: list[str] = []
    evolved_rows = deck.loc[deck["EvolutionDepth"].fillna(0).gt(0) & deck["PreviousStage"].ne("")]
    for parent, group in evolved_rows.groupby("PreviousStage", sort=True):
        child_total = int(group["Copies"].sum())
        if child_total > name_counter[str(parent)]:
            unsupported.extend(group["Card Name"].astype(str).tolist())
    if unsupported:
        raise ValueError(f"Evolution support violated: {unsupported}")
    return {
        "total_cards": int(copies.sum()),
        "role_counts": {key: int(value) for key, value in role_counts.items()},
        "basic_pokemon": basic_count,
        "evolved_pokemon": evolved_count,
        "primary_type_pokemon": primary_count,
        "rule_pokemon": rule_count,
        "trainer_subtypes": {key: int(value) for key, value in subtype_counts.items()},
        "ace_spec": int(deck.loc[deck["IsAceSpec"], "Copies"].sum()),
        "unique_card_ids": int(deck["Card ID"].nunique()),
        "ordinary_name_cap": int(name_counts.max()) if len(name_counts) else 0,
        "evolution_support_pass": True,
    }


def evidence_separation_table() -> pd.DataFrame:
    """Return the immutable epistemic labels used throughout the notebook."""

    return pd.DataFrame(
        [
            ("Catalogue aggregate", "Official competition CSV", "Describes the supplied card pool only"),
            ("Synthetic fixture", "Notebook-owned deterministic state", "Tests mechanics, not game strength"),
            ("Local paired match", "Owned simulator and fixed seeds", "Local evidence, never a Kaggle score"),
            ("Official COMPLETE row", "Exact Kaggle submission row", "Time-specific public rating when nonempty"),
            ("Strategy writeup", "Hackathon artifact", "Judged separately from simulation rating"),
            ("Notebook votes", "Public Code surface", "Raw votes are not medal-eligibility proof"),
        ],
        columns=["Evidence surface", "Source", "Permitted claim"],
    )


def choose_action_fixture(actions: pd.DataFrame) -> dict[str, object]:
    """Apply the documented policy order to a bounded synthetic action table.

    This is a teaching fixture, not a replacement for the official game SDK.
    The columns are deliberately generic so the notebook can test policy order
    without importing or copying another participant's agent.
    """

    required = {
        "action_id",
        "legal",
        "forced",
        "immediate_win",
        "prevents_loss",
        "progress",
        "attack_value",
        "risk",
        "fallback_order",
    }
    missing = required.difference(actions.columns)
    if missing:
        raise ValueError(f"Synthetic action table is missing columns: {sorted(missing)}")
    legal = actions.loc[actions["legal"].astype(bool)].copy()
    if legal.empty:
        return {"action_id": None, "stage": "no_legal_action", "margin": np.nan}

    stage_filters = (
        ("forced", legal["forced"].astype(bool)),
        ("immediate_win", legal["immediate_win"].astype(bool)),
        ("loss_shield", legal["prevents_loss"].astype(bool)),
    )
    for stage, mask in stage_filters:
        candidates = legal.loc[mask].copy()
        if not candidates.empty:
            candidates["policy_score"] = (
                2.0 * pd.to_numeric(candidates["immediate_win"], errors="coerce").fillna(0.0)
                + 1.4 * pd.to_numeric(candidates["prevents_loss"], errors="coerce").fillna(0.0)
                + 0.8 * pd.to_numeric(candidates["progress"], errors="coerce").fillna(0.0)
                + pd.to_numeric(candidates["attack_value"], errors="coerce").fillna(0.0)
                - 0.6 * pd.to_numeric(candidates["risk"], errors="coerce").fillna(0.0)
            )
            candidates = candidates.sort_values(
                ["policy_score", "fallback_order", "action_id"],
                ascending=[False, True, True],
            )
            scores = candidates["policy_score"].to_numpy(float)
            margin = float(scores[0] - scores[1]) if len(scores) > 1 else np.inf
            return {
                "action_id": candidates.iloc[0]["action_id"],
                "stage": stage,
                "margin": margin,
            }

    legal["policy_score"] = (
        0.8 * pd.to_numeric(legal["progress"], errors="coerce").fillna(0.0)
        + pd.to_numeric(legal["attack_value"], errors="coerce").fillna(0.0)
        - 0.6 * pd.to_numeric(legal["risk"], errors="coerce").fillna(0.0)
    )
    legal = legal.sort_values(
        ["policy_score", "fallback_order", "action_id"],
        ascending=[False, True, True],
    )
    scores = legal["policy_score"].to_numpy(float)
    margin = float(scores[0] - scores[1]) if len(scores) > 1 else np.inf
    stage = "value_progress" if np.isfinite(scores[0]) else "deterministic_fallback"
    return {"action_id": legal.iloc[0]["action_id"], "stage": stage, "margin": margin}


__all__ = [
    "DeckQuotas",
    "EXPECTED_COLUMNS",
    "KNOWN_EN_SHA256",
    "build_card_table",
    "classify_card",
    "choose_action_fixture",
    "evidence_separation_table",
    "monte_carlo_basic_and_energy",
    "optimize_demo_deck",
    "pareto_frontier_mask",
    "parse_damage",
    "parse_energy_cost",
    "probability_at_least_one",
    "probability_basic_and_energy",
    "resolve_catalogue_path",
    "score_cards",
    "sha256_file",
    "validate_catalogue",
    "validate_demo_deck",
]


import json
import warnings
from datetime import datetime, timezone
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive.*")
SEED = 20260728
RNG = np.random.default_rng(SEED)
OUTPUT_DIR = Path("ptcg_strategy_atlas_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "ink": "#102A43",
    "blue": "#247BA0",
    "cyan": "#70C1B3",
    "gold": "#F3B61F",
    "coral": "#FF6B6B",
    "paper": "#F7F9FC",
    "muted": "#7A8797",
}
plt.rcParams.update({
    "figure.figsize": (11, 6),
    "figure.facecolor": "white",
    "axes.facecolor": COLORS["paper"],
    "axes.edgecolor": "#D7E0EA",
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelcolor": COLORS["ink"],
    "text.color": COLORS["ink"],
    "font.size": 10,
    "savefig.bbox": "tight",
    "savefig.dpi": 150,
})

def save_figure(name):
    path = OUTPUT_DIR / name
    plt.savefig(path, facecolor="white")
    print(f"saved {path}")
    return path


In [ ]:
catalogue_path = resolve_catalogue_path()
catalogue_sha = sha256_file(catalogue_path)
raw = pd.read_csv(catalogue_path)
contract = validate_catalogue(raw)
cards = score_cards(build_card_table(raw))

snapshot_status = "KNOWN SNAPSHOT" if catalogue_sha == KNOWN_EN_SHA256 else "SCHEMA-VALID NEW SNAPSHOT"
contract_view = pd.DataFrame([
    ("Path", str(catalogue_path)),
    ("SHA256", catalogue_sha),
    ("Snapshot status", snapshot_status),
    ("Move-level rows", f"{contract['rows']:,}"),
    ("Unique Card IDs", f"{contract['unique_card_ids']:,}"),
    ("Move rows beyond unique cards", f"{contract['move_rows_beyond_unique_cards']:,}"),
    ("Columns", contract["columns"]),
], columns=["Contract field", "Observed value"])
display(contract_view)

class_counts = cards["Kind"].value_counts().rename_axis("Card class").reset_index(name="Unique cards")
display(class_counts)
assert cards["Card ID"].is_unique
assert cards["Utility"].notna().all()


In [ ]:
row_class = raw["Stage (Pokémon)/Type (Energy and Trainer)"].map(classify_card)
display_columns = [
    "HP", "Type", "Weakness", "Resistance (Type)", "Retreat",
    "Move Name", "Cost", "Damage", "Effect Explanation", "Previous stage", "Rule",
]
missing_by_class = pd.DataFrame({
    label: raw.loc[row_class.eq(label), display_columns].isna().mean()
    for label in ["Pokemon", "Trainer", "Energy"]
}).T

fig, ax = plt.subplots(figsize=(12, 4.6))
cmap = LinearSegmentedColormap.from_list("missing", ["#E8F5F2", COLORS["gold"], COLORS["coral"]])
image = ax.imshow(missing_by_class.to_numpy(), aspect="auto", cmap=cmap, vmin=0, vmax=1)
ax.set_xticks(range(len(display_columns)), display_columns, rotation=42, ha="right")
ax.set_yticks(range(len(missing_by_class)), missing_by_class.index)
for row in range(missing_by_class.shape[0]):
    for col in range(missing_by_class.shape[1]):
        value = missing_by_class.iat[row, col]
        ax.text(col, row, f"{value:.0%}", ha="center", va="center", fontsize=8,
                color="white" if value > 0.62 else COLORS["ink"])
ax.set_title("Missingness is structural: it changes with card class")
fig.colorbar(image, ax=ax, label="missing share", fraction=0.025)
plt.tight_layout()
save_figure("01_structural_missingness.png")
plt.show()


## Predict — bounded parsing, no invented damage

Energy symbols are counted directly. Damage is accepted only when the field is
an unconditional integer. Multipliers, additions, reductions, and prose remain
explicitly uncertain; downstream efficiency plots never pretend `20×` is 20.


In [ ]:
damage_audit = raw["Damage"].map(parse_damage).apply(pd.Series)
cost_audit = raw["Cost"].map(parse_energy_cost).apply(pd.Series)
damage_counts = damage_audit["damage_kind"].value_counts()
nonempty_cost = cost_audit["cost_text"].ne("")
cost_coverage = float(cost_audit.loc[nonempty_cost, "fully_parsed"].mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1.6, 1]})
damage_counts.sort_values().plot.barh(ax=axes[0], color=COLORS["blue"])
axes[0].set_title("Damage parser coverage")
axes[0].set_xlabel("move rows")
for patch in axes[0].patches:
    axes[0].text(patch.get_width() + 4, patch.get_y() + patch.get_height()/2,
                 f"{int(patch.get_width()):,}", va="center", fontsize=9)
axes[1].bar(["fully parsed", "flagged"], [cost_coverage, 1-cost_coverage],
            color=[COLORS["cyan"], COLORS["coral"]])
axes[1].set_ylim(0, 1.12)
axes[1].set_ylabel("share of nonempty costs")
axes[1].set_title("Visible energy-cost coverage")
for idx, value in enumerate([cost_coverage, 1-cost_coverage]):
    axes[1].text(idx, value + .025, f"{value:.1%}", ha="center", fontweight="bold")
plt.tight_layout()
save_figure("02_bounded_parser_coverage.png")
plt.show()

parser_receipt = pd.DataFrame([
    ("Deterministic integer damage rows", int(damage_counts.get("deterministic", 0))),
    ("Conditional or unparsed damage rows", int(len(damage_audit) - damage_counts.get("deterministic", 0) - damage_counts.get("missing", 0))),
    ("Nonempty energy costs fully parsed", f"{cost_coverage:.2%}"),
], columns=["Parser check", "Observed"])
display(parser_receipt)


## Evaluate — efficiency is a frontier, not a leaderboard

A Pokémon can trade raw pressure for durability, mobility, or easier setup.
The Pareto frontier below keeps cards that are not simultaneously beaten on
HP, deterministic damage-per-energy, and retreat burden. Conditional attacks
remain penalized rather than silently valued.


In [ ]:
pokemon = cards.loc[
    cards["Kind"].eq("Pokemon")
    & cards["HP"].notna()
    & cards["BestDamagePerEnergy"].notna()
    & cards["Retreat"].notna()
].copy()
frontier_values = pokemon[["HP", "BestDamagePerEnergy", "Retreat"]].to_numpy(float)
pokemon["ParetoFrontier"] = pareto_frontier_mask(frontier_values, maximize=(True, True, False))

fig, ax = plt.subplots(figsize=(11, 6.2))
sizes = 35 + 22 * (pokemon["HP"] / pokemon["HP"].median()).clip(.5, 3)
scatter = ax.scatter(
    pokemon["BestDamagePerEnergy"], pokemon["HP"], c=pokemon["Retreat"],
    s=sizes, cmap="viridis_r", alpha=.35, edgecolors="none", label="all eligible cards"
)
frontier = pokemon.loc[pokemon["ParetoFrontier"]]
ax.scatter(frontier["BestDamagePerEnergy"], frontier["HP"], s=90,
           facecolors="none", edgecolors=COLORS["coral"], linewidths=1.7,
           label=f"Pareto frontier ({len(frontier)})")
for _, row in frontier.sort_values("Utility", ascending=False).head(8).iterrows():
    ax.annotate(row["Card Name"], (row["BestDamagePerEnergy"], row["HP"]),
                xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.set_xlabel("best deterministic damage per parsed energy")
ax.set_ylabel("HP")
ax.set_title("Multi-objective card efficiency: pressure, durability, mobility")
ax.legend(frameon=False)
fig.colorbar(scatter, ax=ax, label="retreat burden")
plt.tight_layout()
save_figure("03_pareto_efficiency_frontier.png")
plt.show()


In [ ]:
scenario_pool = cards.loc[
    cards["Kind"].eq("Pokemon") & cards["Type"].isin(["{F}", "{C}"])
].copy()
exemplars = scenario_pool.sort_values("Utility", ascending=False).head(10).copy()
penalties = np.linspace(0, .8, 9)
rank_rows = []
for penalty in penalties:
    score = (
        .34 * exemplars["durability_z"] + .42 * exemplars["pressure_z"]
        + .18 * exemplars["mobility_z"] - penalty * exemplars["conditional_share"]
        - .10 * exemplars["EvolutionDepth"].fillna(0)
    )
    ranks = score.rank(ascending=False, method="min")
    for name, rank in zip(exemplars["Card Name"], ranks):
        rank_rows.append((penalty, name, int(rank)))
rank_stability = pd.DataFrame(rank_rows, columns=["uncertainty_penalty", "Card Name", "rank"])
changed_candidates = int(rank_stability.groupby("Card Name")["rank"].nunique().gt(1).sum())

fig, ax = plt.subplots(figsize=(11, 5.8))
for name, group in rank_stability.groupby("Card Name"):
    ax.plot(group["uncertainty_penalty"], group["rank"], marker="o", linewidth=1.5, label=name)
ax.invert_yaxis()
ax.set_xlabel("penalty on conditional-damage share")
ax.set_ylabel("rank (1 is highest under this scenario)")
if changed_candidates:
    rank_title = f"Rank stability audit: {changed_candidates} candidates swap places"
else:
    rank_title = "Rank stability audit: no swaps in this bounded stress test"
ax.set_title(rank_title)
ax.legend(ncol=2, fontsize=8, frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_figure("04_rank_stability_ribbon.png")
plt.show()


In [ ]:
def first_type_token(value):
    match = re.search(r"\{[^{}]+\}", str(value))
    return match.group(0) if match else ""

type_labels = {
    "{C}": "Colorless", "{D}": "Darkness", "{F}": "Fighting",
    "{G}": "Grass", "{L}": "Lightning", "{M}": "Metal",
    "{P}": "Psychic", "{R}": "Fire", "{W}": "Water",
}

weakness_cards = cards.loc[cards["Kind"].eq("Pokemon")].copy()
weakness_cards["WeaknessToken"] = weakness_cards["Weakness"].map(first_type_token)
weakness_cards = weakness_cards.loc[weakness_cards["Type"].ne("") & weakness_cards["WeaknessToken"].ne("")]
top_types = weakness_cards["Type"].value_counts().head(10).index
top_weakness = weakness_cards["WeaknessToken"].value_counts().head(10).index
weakness_matrix = pd.crosstab(weakness_cards["Type"], weakness_cards["WeaknessToken"]).reindex(
    index=top_types, columns=top_weakness, fill_value=0
)

fig, ax = plt.subplots(figsize=(10, 6))
image = ax.imshow(weakness_matrix.to_numpy(), cmap="Blues", aspect="auto")
ax.set_xticks(
    range(len(top_weakness)), [type_labels.get(token, token) for token in top_weakness],
    rotation=35, ha="right"
)
ax.set_yticks(range(len(top_types)), [type_labels.get(token, token) for token in top_types])
ax.set_xlabel("weak to")
ax.set_ylabel("attacking/card type")
ax.set_title("Weakness coverage map (unique cards, not move rows)")
for row in range(weakness_matrix.shape[0]):
    for col in range(weakness_matrix.shape[1]):
        value = int(weakness_matrix.iat[row, col])
        if value:
            ax.text(col, row, value, ha="center", va="center", fontsize=8,
                    color="white" if value > weakness_matrix.to_numpy().max()*.55 else COLORS["ink"])
fig.colorbar(image, ax=ax, label="unique cards")
plt.tight_layout()
save_figure("05_weakness_coverage_map.png")
plt.show()


## From cards to sixty — a transparent integer portfolio

The following is a **teaching scenario**, not a claim of optimal play. We choose
Fighting as the primary type and require exactly 60 cards with explicit role
quotas, name-level copy caps, unlimited matching Basic Energy, supported
evolution chains, a bounded number of rule-box Pokémon, and Trainer subtype
coverage. Every constraint is independently checked after optimization.


In [ ]:
PRIMARY_TYPE = "{F}"
deck = optimize_demo_deck(cards, primary_type=PRIMARY_TYPE)
deck_receipt = validate_demo_deck(deck)
deck_export_columns = [
    "Card ID", "Card Name", "Kind", "StageOrTrainerType", "PreviousStage",
    "Type", "Rule", "Copies", "Utility", "PrimaryTypeScenario",
]
deck[deck_export_columns].to_csv(OUTPUT_DIR / "strategy_demo_deck.csv", index=False)
display(pd.DataFrame([(key, value) for key, value in deck_receipt.items()],
                     columns=["Independent deck check", "Observed"]))
display(deck[deck_export_columns].reset_index(drop=True))

role_counts = deck.groupby("Kind")["Copies"].sum().reindex(["Pokemon", "Trainer", "Energy"])
fig, axes = plt.subplots(1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [1, 1.8]})
axes[0].pie(role_counts, labels=role_counts.index, autopct="%1.0f%%",
            colors=[COLORS["coral"], COLORS["blue"], COLORS["gold"]],
            wedgeprops={"width": .42, "edgecolor": "white"})
axes[0].set_title("The 60-card role budget")
deck_plot = deck.sort_values(["Kind", "Copies", "Utility"], ascending=[True, True, True])
bar_colors = deck_plot["Kind"].map({"Pokemon": COLORS["coral"], "Trainer": COLORS["blue"], "Energy": COLORS["gold"]})
axes[1].barh(deck_plot["Card Name"], deck_plot["Copies"], color=bar_colors)
axes[1].set_xlabel("copies")
axes[1].set_title("Every copy satisfies the explicit portfolio constraints")
axes[1].set_xlim(0, max(15, int(deck_plot["Copies"].max()) + 1))
plt.tight_layout()
save_figure("06_deck_role_and_copy_audit.png")
plt.show()


## Opening consistency — exact first, simulation second

For disjoint Basic-Pokémon and Energy roles, inclusion–exclusion gives the exact
probability that a seven-card opening hand contains at least one of each. A
seeded Monte Carlo estimate is used only as a cross-check; disagreement fails
the notebook rather than being explained away.


In [ ]:
basic_selected = int(deck.loc[deck["IsBasicPokemon"], "Copies"].sum())
energy_selected = int(deck.loc[deck["Kind"].eq("Energy"), "Copies"].sum())
exact_opening = probability_basic_and_energy(basic_selected, energy_selected)
mc_opening = monte_carlo_basic_and_energy(
    basic_selected, energy_selected, trials=160_000, seed=SEED
)
opening_delta = abs(exact_opening - mc_opening)
assert opening_delta < .008, (exact_opening, mc_opening, opening_delta)

basic_grid = np.arange(6, 17)
energy_grid = np.arange(8, 19)
risk_surface = np.array([
    [probability_basic_and_energy(int(basic), int(energy)) for energy in energy_grid]
    for basic in basic_grid
])
fig, ax = plt.subplots(figsize=(11, 6))
image = ax.imshow(risk_surface, origin="lower", aspect="auto", cmap="YlGnBu", vmin=.35, vmax=.92)
ax.set_xticks(range(len(energy_grid)), energy_grid)
ax.set_yticks(range(len(basic_grid)), basic_grid)
ax.set_xlabel("Energy cards in 60")
ax.set_ylabel("Basic Pokémon in 60")
ax.set_title("Exact probability of opening with both a Basic and Energy")
chosen_x = int(np.where(energy_grid == energy_selected)[0][0]) if energy_selected in energy_grid else None
chosen_y = int(np.where(basic_grid == basic_selected)[0][0]) if basic_selected in basic_grid else None
if chosen_x is not None and chosen_y is not None:
    ax.scatter(chosen_x, chosen_y, s=180, facecolors="none", edgecolors=COLORS["coral"], linewidths=2.5)
    ax.text(chosen_x+.25, chosen_y+.15, f"scenario {exact_opening:.1%}", color=COLORS["coral"], fontweight="bold")
fig.colorbar(image, ax=ax, label="exact probability")
plt.tight_layout()
save_figure("07_opening_consistency_surface.png")
plt.show()

display(pd.DataFrame([
    ("Basic Pokémon copies", basic_selected),
    ("Energy copies", energy_selected),
    ("Exact joint opening probability", f"{exact_opening:.4%}"),
    ("Seeded Monte Carlo estimate", f"{mc_opening:.4%}"),
    ("Absolute cross-check delta", f"{opening_delta:.6f}"),
], columns=["Opening check", "Observed"]))


## From sixty to decisions — order before score

The control policy is intentionally boring where reliability matters:

`legal actions → forced actions → immediate wins → loss shields → value/progress → deterministic fallback`

Synthetic fixtures test that order without importing the official SDK or any
competitor's policy. They reveal how often each gate decides the action and how
large the winning action margin is.


In [ ]:
decision_rows = []
for state_id in range(500):
    action_count = int(RNG.integers(2, 9))
    actions = pd.DataFrame({
        "action_id": [f"s{state_id}_a{idx}" for idx in range(action_count)],
        "legal": RNG.random(action_count) > .12,
        "forced": False,
        "immediate_win": RNG.random(action_count) < .04,
        "prevents_loss": RNG.random(action_count) < .10,
        "progress": RNG.beta(2, 2, action_count),
        "attack_value": RNG.beta(2, 2, action_count),
        "risk": RNG.beta(1.5, 3, action_count),
        "fallback_order": np.arange(action_count),
    })
    if state_id % 125 == 0:
        # Deterministic fail-closed fixtures: no action is legal in these states.
        actions["legal"] = False
    if not actions["legal"].any():
        # Preserve a no-legal-action fixture rather than silently repairing every state.
        pass
    elif RNG.random() < .06:
        legal_indices = actions.index[actions["legal"]]
        actions.loc[int(RNG.choice(legal_indices)), "forced"] = True
    result = choose_action_fixture(actions)
    decision_rows.append((state_id, result["stage"], result["margin"], int(actions["legal"].sum())))

decisions = pd.DataFrame(decision_rows, columns=["state_id", "stage", "margin", "legal_actions"])
stage_order = ["forced", "immediate_win", "loss_shield", "value_progress", "no_legal_action"]
stage_counts = decisions["stage"].value_counts().reindex(stage_order, fill_value=0)
finite_margins = decisions.loc[np.isfinite(decisions["margin"]), "margin"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
stage_counts.plot.bar(ax=axes[0], color=[COLORS["gold"], COLORS["coral"], COLORS["cyan"], COLORS["blue"], COLORS["muted"]])
axes[0].set_title("Which gate selected the action?")
axes[0].set_ylabel("synthetic states")
axes[0].tick_params(axis="x", rotation=30)
axes[1].hist(finite_margins.clip(upper=finite_margins.quantile(.98)), bins=28, color=COLORS["blue"], alpha=.85)
axes[1].axvline(finite_margins.median(), color=COLORS["coral"], linestyle="--",
                label=f"median {finite_margins.median():.3f}")
axes[1].set_title("Chosen-action margin: low values signal uncertainty")
axes[1].set_xlabel("policy-score margin (98th-percentile clipped)")
axes[1].legend(frameon=False)
plt.tight_layout()
save_figure("08_policy_gate_and_margin_audit.png")
plt.show()

assert (decisions["stage"] == "no_legal_action").any()
assert set(decisions["stage"]).issubset(set(stage_order))


## Catch — local evidence is not an official rating

Simulation ratings are stochastic and move as evaluation episodes accumulate.
The plot below is a historical receipt-bound snapshot, not a current-highest
claim. It shows why a single episode batch must never be presented as a stable
score improvement.


In [ ]:
score_history = pd.DataFrame([
    (55056992, "2026-07-28T14:18:41Z", 920.0, "Observable Meta Router V3"),
    (55056992, "2026-07-28T14:25:32Z", 912.4, "Observable Meta Router V3"),
    (55056992, "2026-07-28T14:27:35Z", 848.6, "Observable Meta Router V3"),
    (55056992, "2026-07-28T14:33:11Z", 862.1, "Observable Meta Router V3"),
    (55056992, "2026-07-28T14:34:53Z", 862.1, "Observable Meta Router V3"),
    (55055028, "2026-07-28T14:19:46Z", 818.0, "Owned V21 V5"),
    (55055028, "2026-07-28T14:25:32Z", 798.3, "Owned V21 V5"),
    (55055028, "2026-07-28T14:34:53Z", 798.3, "Owned V21 V5"),
    (55057450, "2026-07-28T14:27:35Z", 706.1, "V12 probabilistic search V10"),
    (55057450, "2026-07-28T14:33:11Z", 784.3, "V12 probabilistic search V10"),
    (55057450, "2026-07-28T14:34:53Z", 784.3, "V12 probabilistic search V10"),
], columns=["row_id", "observed_at_utc", "publicScore", "label"])
score_history["observed_at_utc"] = pd.to_datetime(score_history["observed_at_utc"], utc=True)

fig, ax = plt.subplots(figsize=(11, 5.5))
for (row_id, label), group in score_history.groupby(["row_id", "label"]):
    group = group.sort_values("observed_at_utc")
    ax.plot(group["observed_at_utc"], group["publicScore"], marker="o", linewidth=2,
            label=f"row {row_id} — {label}")
ax.set_ylabel("historical observed publicScore")
ax.set_xlabel("receipt timestamp (UTC)")
ax.set_title("Official simulation ratings can drift after COMPLETE")
ax.legend(fontsize=8, frameon=False)
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
save_figure("09_official_score_drift_receipt.png")
plt.show()

score_history.to_csv(OUTPUT_DIR / "historical_score_drift_receipt.csv", index=False)


## Public research attribution — mechanisms, not source code

The following public notebooks were inspected to understand what readers find
useful. Their code, prose, tables, deck lists, and outputs were not copied. No
inspected source contained an embedded SPDX or full named-license notice, so
this notebook uses attribution plus independent implementation.


In [ ]:
attribution = pd.DataFrame([
    ("Nur Srijan", "nursrijan/pokemon-tcg-eda-deck-engine", 97,
     "Catalogue-to-deck narrative", "No code copied"),
    ("Jeki Wan Taufik", "jek1wantaufik/pok-mon-tcg-ai-strategy-analysis", 59,
     "Separate deck philosophy, decision engine, robustness", "No code copied"),
    ("Gowri Shankar Penugonda", "gowrishankarp/pok-mon-tcg-power-creep-damage", 21,
     "Longitudinal efficiency communication", "No code copied"),
    ("Himanshu Dhiman", "hmnshudhmn24/pok-mon-tcg-ai-battle-challenge-ppo-agent", 19,
     "Agent architecture attracts interest; explanation gap remains", "No code copied"),
    ("Rahul Jiwane", "rahuljiwane/pokemon-tcg-rahul-jiwane-10", 6,
     "Recent audience signal; outline overlaps an older public notebook", "Excluded from implementation"),
], columns=["Author", "Public slug", "Raw vote snapshot", "Idea examined", "Reuse status"])
display(attribution)
attribution.to_csv(OUTPUT_DIR / "source_attribution.csv", index=False)

evidence = evidence_separation_table()
display(evidence)
evidence.to_csv(OUTPUT_DIR / "evidence_separation.csv", index=False)


## What survived falsification?

- Conditional damage stayed conditional; it was never cast to a convenient
  number.
- The portfolio contains exactly 60 cards and passes an independent
  scenario-constraint validator.
- Opening consistency agrees between an exact formula and seeded simulation.
- The decision policy explicitly exposes no-legal-action and low-margin states.
- Official rating drift is reported as a historical observation, not a stable
  superlative.
- Public notebooks supplied ideas to examine, not code to transplant.

The useful next experiment is a seat-balanced, replay-disjoint simulator A/B
using only provenance-cleared owned policies. That experiment belongs to the
simulation track and must remain separate from this strategy notebook.


In [ ]:
figure_files = sorted(path for path in OUTPUT_DIR.glob("*.png") if path.name[:2].isdigit())
assert len(figure_files) == 9, [path.name for path in figure_files]
for path in figure_files:
    assert path.stat().st_size > 10_000, (path, path.stat().st_size)

text_outputs = list(OUTPUT_DIR.glob("*.csv")) + list(OUTPUT_DIR.glob("*.json"))
for path in text_outputs:
    assert "Traceback (most recent call last)" not in path.read_text(encoding="utf-8", errors="ignore")

artifact_hashes = {
    path.name: sha256_file(path)
    for path in sorted(OUTPUT_DIR.iterdir())
    if path.is_file() and path.name != "validation_receipt.json"
}
final_receipt = {
    "receipt_type": "ptcg_strategy_decision_atlas_local_notebook_execution",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "catalogue_sha256": catalogue_sha,
    "catalogue_snapshot_status": snapshot_status,
    "catalogue_contract": contract,
    "card_level_rows": int(len(cards)),
    "deck_validation": deck_receipt,
    "opening_exact": exact_opening,
    "opening_monte_carlo": mc_opening,
    "opening_absolute_delta": opening_delta,
    "synthetic_decision_states": int(len(decisions)),
    "figure_count": len(figure_files),
    "artifact_sha256": artifact_hashes,
    "public_research_reuse": "mechanism-only; no code copied",
    "competition_submission_created": False,
    "official_score_claim": False,
    "validation_status": "PASS",
}
(OUTPUT_DIR / "validation_receipt.json").write_text(
    json.dumps(final_receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print(json.dumps(final_receipt, indent=2, sort_keys=True))


## Stable-public filter refresh — why this notebook says *no* so often

This public Strategy MRI is not a scoring notebook. It is the campaign’s
readable evidence layer: exact completed rows, stable public/meta ideas, and
the local gates that stop attractive but fragile ideas from becoming noisy
submissions.

The 2026-07-30 refresh adds two things readers can audit:

- a latest-observed official-row snapshot with drifted Pokémon scores; and
- a falsification dashboard for stable-looking microlevers that failed broad
  local screens.

No `submission.tar.gz` is written here. No official competition submission is
created. The notebook identity is preserved.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("ptcg_strategy_atlas_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CURRENT_SCORE_ROWS = json.loads(r'''[
  {
    "public_score": 844.4,
    "role": "owned high-water",
    "row_id": 55056992,
    "status": "COMPLETE",
    "surface": "Observable meta-router V3"
  },
  {
    "public_score": 806.6,
    "role": "strong fallback",
    "row_id": 55057450,
    "status": "COMPLETE",
    "surface": "V12 probabilistic search"
  },
  {
    "public_score": 798.3,
    "role": "startup repair",
    "row_id": 55055028,
    "status": "COMPLETE",
    "surface": "V21 faithful-loader fix"
  },
  {
    "public_score": 782.7,
    "role": "independent surface",
    "row_id": 55057790,
    "status": "COMPLETE",
    "surface": "RMY/Souta loader"
  },
  {
    "public_score": 781.7,
    "role": "latest held branch",
    "row_id": 55094054,
    "status": "COMPLETE",
    "surface": "Observable V22 contextual mill"
  },
  {
    "public_score": 768.3,
    "role": "drifted branch",
    "row_id": 55086222,
    "status": "COMPLETE",
    "surface": "Observable V19 Articuno tech"
  },
  {
    "public_score": 678.3,
    "role": "weak branch",
    "row_id": 55058197,
    "status": "COMPLETE",
    "surface": "Observable router V4"
  },
  {
    "public_score": 555.3,
    "role": "historical floor",
    "row_id": 55000147,
    "status": "COMPLETE",
    "surface": "Legacy visible router"
  },
  {
    "public_score": 531.3,
    "role": "false lead",
    "row_id": 55078864,
    "status": "COMPLETE",
    "surface": "RMY V4a Azumarill"
  }
]''')
CANDIDATE_GATES = json.loads(r'''[
  {
    "candidate": "Flutter Mane active-skill denial",
    "gate": "HOLD",
    "local_delta": 0.0,
    "reason": "faithful-loader PASS but zero action divergences",
    "receipt": "flutter_active_skill_deny_20260730T0253Z"
  },
  {
    "candidate": "Enhanced Hammer special target",
    "gate": "HOLD",
    "local_delta": -0.0125,
    "reason": "16-game/anchor confirmation negative with eight regressions",
    "receipt": "hammer_special_energy_target_v1_16g_seed202607300325"
  },
  {
    "candidate": "Prize-clock optional thinning guard",
    "gate": "HOLD",
    "local_delta": -0.05,
    "reason": "negative aggregate and broad anchor regressions",
    "receipt": "prize_clock_optional_thin_guard_v1_8g_seed202607300315"
  },
  {
    "candidate": "Tarountula/Roserade field weighting",
    "gate": "HOLD",
    "local_delta": 0.00109,
    "reason": "Rocket-Spidops gain offset by exact-Garchomp regression",
    "receipt": "FINAL_JULY26_FIELD_WEIGHTING_RECEIPT_V2"
  }
]''')
EXTERNAL_META_SIGNALS = json.loads(r'''[
  {
    "safe_translation": "only patch a branch when it creates action divergence and survives anchor tests",
    "signal": "visible-threat-gated tech is safer than unconditional deck rewrites",
    "source": "Mature Alakazam tech notebook"
  },
  {
    "safe_translation": "separate specialist levers from broad promotion candidates",
    "signal": "large specialist gains can be erased by a single high-share counter-anchor",
    "source": "Roserade/Tarountula proxy audit"
  },
  {
    "safe_translation": "show exact row IDs and latest observed scores instead of title scores",
    "signal": "several recent COMPLETE rows drifted after early observation",
    "source": "Official score drift snapshot"
  },
  {
    "safe_translation": "require broad no-regression local screens before any public scorer promotion",
    "signal": "the current high-water is robust enough that small selector fixes can regress it",
    "source": "844 Observable control line"
  }
]''')
REFRESH_UTC = "2026-07-30T03:20Z"

score_rows = pd.DataFrame(CURRENT_SCORE_ROWS)
score_rows["delta_vs_floor"] = score_rows["public_score"] - float(score_rows["public_score"].min())
score_rows["delta_vs_highwater"] = score_rows["public_score"] - float(score_rows["public_score"].max())
score_rows.to_csv(OUTPUT_DIR / "strategy_mri_current_pokemon_score_rows_20260730.csv", index=False)
display(score_rows)

gate_df = pd.DataFrame(CANDIDATE_GATES)
gate_df.to_csv(OUTPUT_DIR / "strategy_mri_stable_filter_candidate_gates_20260730.csv", index=False)
display(gate_df)

meta_df = pd.DataFrame(EXTERNAL_META_SIGNALS)
meta_df.to_csv(OUTPUT_DIR / "strategy_mri_external_meta_sources_20260730.csv", index=False)
display(meta_df)

palette = {
    "ink": "#111827",
    "slate": "#64748b",
    "blue": "#2563eb",
    "green": "#16a34a",
    "orange": "#f97316",
    "red": "#dc2626",
    "purple": "#7c3aed",
}

fig, ax = plt.subplots(figsize=(13.0, 5.1))
plot_rows = score_rows.sort_values("public_score", ascending=False)
colors = [palette["green"] if role == "owned high-water" else palette["blue"] for role in plot_rows["role"]]
bars = ax.bar(plot_rows["surface"], plot_rows["public_score"], color=colors)
ax.set_title("Latest observed exact COMPLETE rows: score drift is part of the evidence", loc="left", fontsize=14.5, fontweight="bold")
ax.set_ylabel("publicScore")
ax.set_ylim(500, 880)
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.22)
for bar, row in zip(bars, plot_rows.to_dict("records")):
    ax.text(bar.get_x() + bar.get_width() / 2, row["public_score"] + 7, f"{row['public_score']:.1f}\n#{int(row['row_id'])}", ha="center", fontsize=8.0)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "strategy_mri_exact_official_rows_20260730.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(12.0, 4.8))
plot_df = score_rows.sort_values("delta_vs_floor")
ax.barh(plot_df["surface"], plot_df["delta_vs_floor"], color=[palette["green"] if r == "owned high-water" else palette["slate"] for r in plot_df["role"]])
ax.set_title("What survives the floor: high-water, fallbacks, and false leads", loc="left", fontsize=14.5, fontweight="bold")
ax.set_xlabel("publicScore gain vs current floor row")
for y, row in enumerate(plot_df.to_dict("records")):
    ax.text(row["delta_vs_floor"] + 5, y, f"+{row['delta_vs_floor']:.1f}", va="center", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.22)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "strategy_mri_gain_vs_floor_20260730.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(12.8, 4.8))
gate_order = gate_df.assign(color=gate_df["local_delta"].map(lambda x: palette["red"] if x < 0 else palette["orange"]))
ax.barh(gate_order["candidate"], gate_order["local_delta"], color=gate_order["color"])
ax.axvline(0, color=palette["ink"], lw=1.2)
ax.axvline(0.03, color=palette["green"], lw=1.2, ls="--", label="promotion floor")
ax.set_title("Stable-looking levers must change decisions and clear broad anchors", loc="left", fontsize=14.5, fontweight="bold")
ax.set_xlabel("local aggregate delta vs 844 control")
for y, row in enumerate(gate_order.to_dict("records")):
    ax.text(row["local_delta"] + (0.004 if row["local_delta"] >= 0 else -0.004), y, row["gate"], va="center", ha="left" if row["local_delta"] >= 0 else "right", fontsize=9, fontweight="bold")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "strategy_mri_stable_filter_candidate_gates_20260730.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(13.6, 5.1))
ax.axis("off")
cards = [
    ("public idea", "stable source\nnot title-score lore", palette["purple"]),
    ("tiny patch", "narrow trigger\nno deck identity drift", palette["blue"]),
    ("action trace", "must alter decisions\nnot proxy noise", palette["orange"]),
    ("broad screen", "many anchors\nno material regression", palette["red"]),
    ("Kaggle run", "exact output\nthen UI-only submit", palette["green"]),
]
for i, (head, body, color) in enumerate(cards):
    x = 0.08 + i * 0.215
    ax.text(x, 0.70, head, ha="center", va="center", color="white", fontsize=10.2, fontweight="bold", transform=ax.transAxes, bbox=dict(boxstyle="round,pad=.55", fc=color, ec="none"))
    ax.text(x, 0.40, body, ha="center", va="center", color=palette["ink"], fontsize=9.1, transform=ax.transAxes)
    if i < len(cards) - 1:
        ax.annotate("", xy=(x + 0.126, 0.70), xytext=(x + 0.087, 0.70), xycoords=ax.transAxes, arrowprops=dict(arrowstyle="->", lw=1.8, color=palette["slate"]))
ax.set_title("Stable-public filter: beautiful notebooks should reduce noisy submissions", loc="left", fontsize=14.5, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "strategy_mri_stable_filter_pipeline_20260730.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(13.2, 5.0))
ax.axis("off")
for i, row in enumerate(meta_df.to_dict("records")):
    y = 0.82 - i * 0.20
    ax.text(0.02, y, row["source"], ha="left", va="center", color="white", fontsize=9.2, fontweight="bold", transform=ax.transAxes, bbox=dict(boxstyle="round,pad=.45", fc=palette["blue"], ec="none"))
    ax.text(0.34, y + 0.035, row["signal"], ha="left", va="center", color=palette["ink"], fontsize=9.1, transform=ax.transAxes)
    ax.text(0.34, y - 0.045, row["safe_translation"], ha="left", va="center", color=palette["slate"], fontsize=8.8, transform=ax.transAxes)
ax.set_title("Meta signal translation: public ideas become gates before they become submissions", loc="left", fontsize=14.5, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "strategy_mri_external_meta_router_map_20260730.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(13.4, 4.6))
ax.axis("off")
evidence_cards = [
    ("COMPLETE rows", "exact row IDs\nnonempty publicScore", palette["green"]),
    ("candidate holds", "saved receipts\nzero Kaggle mutation", palette["orange"]),
    ("identity guard", "same id/title/code_file\nno deletion", palette["blue"]),
    ("next run", "only after local pass\nand Kaggle output validation", palette["purple"]),
]
for i, (head, body, color) in enumerate(evidence_cards):
    x = 0.12 + i * 0.25
    ax.text(x, 0.66, head, ha="center", va="center", color="white", fontsize=10.2, fontweight="bold", transform=ax.transAxes, bbox=dict(boxstyle="round,pad=.55", fc=color, ec="none"))
    ax.text(x, 0.34, body, ha="center", va="center", color=palette["ink"], fontsize=9.0, transform=ax.transAxes)
ax.set_title("Evidence boundary: public visualization, not a scorer or submission artifact", loc="left", fontsize=14.5, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "strategy_mri_evidence_chain_20260730.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
current_visuals = sorted(str(path.relative_to(OUTPUT_DIR)) for path in OUTPUT_DIR.glob("*20260730*.png"))
refresh_receipt = {
    "status": "PUBLIC_STRATEGY_MRI_STABLE_FILTER_REFRESH_PASS",
    "notebook_slug": "prvsiyan/ptcg-strategy-mri-2-022-cards-to-60-decisions",
    "refresh_utc": REFRESH_UTC,
    "scope": "CPU-only public strategy notebook; latest official-row drift and local candidate falsification addendum; no submission archive",
    "exact_official_rows": CURRENT_SCORE_ROWS,
    "candidate_gates": CANDIDATE_GATES,
    "external_meta_signals": EXTERNAL_META_SIGNALS,
    "visual_count": len(current_visuals),
    "visual_files": current_visuals,
    "submission_artifact_written": False,
    "identity_guard": {
        "id": "prvsiyan/ptcg-strategy-mri-2-022-cards-to-60-decisions",
        "title": "PTCG Strategy MRI | 2,022 Cards to 60 Decisions",
        "code_file": "ptcg-strategy-mri-2-022-cards-to-60-decisions.ipynb",
        "delete_action_allowed": False,
    },
}
(OUTPUT_DIR / "strategy_mri_refresh_receipt_20260730.json").write_text(json.dumps(refresh_receipt, indent=2, sort_keys=True) + "\n")
assert len(current_visuals) == 6, current_visuals
assert not Path("submission.tar.gz").exists()
print(json.dumps({"status": "PASS", "visual_count": len(current_visuals), "submission_artifact_written": False}, indent=2))


## External meta refresh — translating public Pocket signals into safer 60-card heuristics

This Strategy MRI is still a public evidence notebook, not a scoring archive.
The addendum below uses current external Pokémon TCG Pocket metagame sources as
**idea-level context**: tournament usage, tier-list stability, and resource
patterns such as energy acceleration, bench pressure, retreat reset, and
anti-ex counterplay.

The point is not to copy a 20-card Pocket deck into a 60-card Kaggle deck. The
point is to keep the deck-building narrative alive for readers: modern good
decks tend to win by making resources convertible under pressure, not by simply
sorting cards by damage.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

PTCG_STRATEGY_MRI_EXTERNAL_META_V5_REFRESH_MARKER = "external-meta-heuristic-transfer-v5"
SNAPSHOT_UTC = "2026-08-01T05:45:08Z"
v5_dir = Path("ptcg_strategy_atlas_outputs")
v5_dir.mkdir(exist_ok=True)

colors = {
    "ink": "#102A43",
    "muted": "#62748A",
    "blue": "#2563EB",
    "cyan": "#06B6D4",
    "green": "#16A34A",
    "gold": "#F59E0B",
    "red": "#DC2626",
    "violet": "#7C3AED",
    "paper": "#F8FAFC",
    "line": "#CBD5E1",
}
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": colors["paper"],
    "axes.edgecolor": colors["line"],
    "axes.labelcolor": colors["ink"],
    "xtick.color": colors["ink"],
    "ytick.color": colors["ink"],
    "text.color": colors["ink"],
    "font.size": 10,
    "axes.titleweight": "bold",
    "savefig.bbox": "tight",
    "savefig.dpi": 150,
})

external_sources = pd.DataFrame([
    {
        "source": "PTCGPocket.gg tier list",
        "url": "https://ptcgpocket.gg/tier-list/",
        "snapshot_detail": "Updated 2026-07-15; lists S/A/B tiers for Everyday Wonders.",
        "idea_signal": "Successful archetypes are described by engine roles: acceleration, sniping, status, retreat, and backup attackers.",
        "reuse_boundary": "Public prose/meta context only; no code or decklist copied into Kaggle scorer.",
    },
    {
        "source": "Pokemon Zone data-driven tier list",
        "url": "https://www.pokemon-zone.com/decks/",
        "snapshot_detail": "Ruler of the Skies snapshot: 6,473 matches, 13 tournaments, 39 established decks.",
        "idea_signal": "Wilson-bound tiers reward sample reliability, not raw spike win rate.",
        "reuse_boundary": "Use as vote-friendly evidence weighting; no implementation imported.",
    },
    {
        "source": "Limitless Pocket decks",
        "url": "https://play.limitlesstcg.com/decks?game=pocket",
        "snapshot_detail": "Observed 11 tournaments, 1,121 players, and 2,998 matches in the current page snapshot.",
        "idea_signal": "Usage share and win rate can diverge; popularity is not the same as expected value.",
        "reuse_boundary": "Aggregate context only; no match data redistribution.",
    },
    {
        "source": "Lean metagame case study",
        "url": "https://arxiv.org/abs/2607.08692",
        "snapshot_detail": "Formal 2026 TCG metagame analysis argues for explicit trust boundaries around empirical claims.",
        "idea_signal": "A deck can be popular but strategically dominated under equilibrium-style stress.",
        "reuse_boundary": "Methodological framing only.",
    },
])
external_sources.to_csv(v5_dir / "ptcg_strategy_mri_v5_external_source_map.csv", index=False)

meta_decks = pd.DataFrame([
    {"archetype": "Suicune ex Baxcalibur", "source": "PTCGPocket.gg", "tier": "S", "primary_resource": "Water acceleration + draw", "transfer": "value draw/energy consistency over raw damage"},
    {"archetype": "Miraidon ex Magnezone", "source": "PTCGPocket.gg", "tier": "S", "primary_resource": "Lightning banked-energy burst", "transfer": "score cards for stored-energy conversion"},
    {"archetype": "Mega Blaziken ex Greninja", "source": "PTCGPocket.gg", "tier": "S", "primary_resource": "burn/stadium + bench ping", "transfer": "reward non-active pressure and damage routing"},
    {"archetype": "Zoroark ex Mega Absol ex", "source": "PTCGPocket.gg", "tier": "S", "primary_resource": "cheap attacks + retreat reset", "transfer": "prefer low-cost pivots when damage is sufficient"},
    {"archetype": "Mega Altaria ex Espeon", "source": "Pokemon Zone / PTCGPocket.gg", "tier": "A/S", "primary_resource": "status stall + psychic pressure", "transfer": "value tempo denial separately from HP/DPE"},
    {"archetype": "Giratina ex Oricorio", "source": "PTCGPocket.gg", "tier": "A", "primary_resource": "self-charge + ex immunity counterplay", "transfer": "add opponent-ex exposure and anti-ex utility"},
    {"archetype": "Hydreigon Mega Absol ex", "source": "Limitless / Pokemon Zone", "tier": "B context", "primary_resource": "self-charge dark engine", "transfer": "separate popularity from confidence-adjusted value"},
])
meta_decks.to_csv(v5_dir / "ptcg_strategy_mri_v5_meta_deck_signals.csv", index=False)

heuristics = pd.DataFrame([
    {
        "heuristic": "engine-first consistency",
        "why_it_matters": "Modern lists keep an attacker online by draw, search, or energy acceleration.",
        "catalogue_proxy": "opening Basic+Energy probability; lower setup variance",
        "risk_if_overused": "overfits to solitaire setup and loses to pressure",
        "weight_hint": 0.24,
    },
    {
        "heuristic": "stored-energy conversion",
        "why_it_matters": "Banked energy turns board state into burst damage or tempo.",
        "catalogue_proxy": "energy cost, deterministic damage per energy, retreat",
        "risk_if_overused": "expensive attackers can brick if draw support is missing",
        "weight_hint": 0.18,
    },
    {
        "heuristic": "bench/reach pressure",
        "why_it_matters": "Greninja-style pings and snipes convert incomplete knockouts into future turns.",
        "catalogue_proxy": "effect text tokens: bench, anywhere, damage counters",
        "risk_if_overused": "conditional text can be misread as guaranteed damage",
        "weight_hint": 0.17,
    },
    {
        "heuristic": "anti-ex and prize-denial counterplay",
        "why_it_matters": "Cards such as Oricorio-like counters punish overreliance on ex attackers.",
        "catalogue_proxy": "rule/ex markers and effect text that changes target legality",
        "risk_if_overused": "dead tech slots into non-ex or speed-heavy fields",
        "weight_hint": 0.15,
    },
    {
        "heuristic": "pivot and reset value",
        "why_it_matters": "Retreat/reset tools preserve a winning board before it becomes a prize liability.",
        "catalogue_proxy": "retreat cost and switch/return/heal text features",
        "risk_if_overused": "too many pivots dilute attackers and energy",
        "weight_hint": 0.14,
    },
    {
        "heuristic": "popularity skepticism",
        "why_it_matters": "Usage is an exposure prior, not proof of matchup value.",
        "catalogue_proxy": "confidence-adjusted public signal, not raw rank",
        "risk_if_overused": "ignores real field share and tech frequency",
        "weight_hint": 0.12,
    },
])
assert math.isclose(float(heuristics.weight_hint.sum()), 1.0, rel_tol=0, abs_tol=1e-12)
heuristics.to_csv(v5_dir / "ptcg_strategy_mri_v5_resource_heuristic_matrix.csv", index=False)

official_rows = pd.DataFrame([
    {"row": 55131346, "status": "SubmissionStatus.COMPLETE", "publicScore": 660.8, "claim_role": "latest observed complete row", "action": "do not resubmit unchanged"},
    {"row": 55094054, "status": "SubmissionStatus.COMPLETE", "publicScore": 779.1, "claim_role": "complete context", "action": "context only"},
    {"row": 55086222, "status": "SubmissionStatus.COMPLETE", "publicScore": 769.2, "claim_role": "complete context", "action": "context only"},
    {"row": 55056992, "status": "SubmissionStatus.COMPLETE", "publicScore": 844.4, "claim_role": "own high-water complete row", "action": "claim exact row only"},
    {"row": 55078668, "status": "SubmissionStatus.ERROR", "publicScore": np.nan, "claim_role": "not score evidence", "action": "ignore score"},
])
official_rows.to_csv(v5_dir / "ptcg_strategy_mri_v5_official_score_boundary.csv", index=False)

leaderboard_context = pd.DataFrame([
    {"rank": 1, "teamName": "Majkel1337", "score": 1252.8, "submissionDate": "2026-07-31 21:33:01.976000"},
    {"rank": 2, "teamName": "flg", "score": 1181.7, "submissionDate": "2026-07-27 16:13:32.776000"},
    {"rank": 3, "teamName": "Sixth Sense", "score": 1177.5, "submissionDate": "2026-07-31 13:22:03.726000"},
    {"rank": 4, "teamName": "keidroid", "score": 1159.3, "submissionDate": "2026-08-01 04:10:53.883000"},
    {"rank": 5, "teamName": "James Cox & Henry Chao", "score": 1145.7, "submissionDate": "2026-07-31 11:11:08.483000"},
])
leaderboard_context.to_csv(v5_dir / "ptcg_strategy_mri_v5_public_leaderboard_gap.csv", index=False)

vote_runway = pd.DataFrame([
    {"slug": "ptcg-alakazam-2-304-game-audit-12-visuals", "votes": 23, "lane": "already silver-watch/high evidence"},
    {"slug": "ptcg-844-4-observable-meta-router-visuals", "votes": 16, "lane": "near silver threshold"},
    {"slug": "ptcg-ai-battle-search-audited-alakazam-v9", "votes": 14, "lane": "near silver threshold"},
    {"slug": "ptcg-ai-battle-search-audited-alakazam-v12", "votes": 11, "lane": "needs fresher public context"},
    {"slug": "ptcg-ai-battle-visible-grim-belief-alakazam-v21", "votes": 10, "lane": "needs stronger reader story"},
    {"slug": "ptcg-strategy-mri-2-022-cards-to-60-decisions", "votes": 7, "lane": "current V5 refresh target"},
])
vote_runway.to_csv(v5_dir / "ptcg_strategy_mri_v5_vote_runway.csv", index=False)

fig, ax = plt.subplots(figsize=(12.5, 5.8))
ordered = heuristics.sort_values("weight_hint")
ax.barh(ordered.heuristic, ordered.weight_hint, color=colors["blue"], alpha=0.88)
ax.set_title("V5 heuristic transfer: what external meta changes in the MRI")
ax.set_xlabel("normalized heuristic budget")
for y, row in enumerate(ordered.itertuples()):
    ax.text(row.weight_hint + 0.006, y, row.catalogue_proxy, va="center", fontsize=8.5, color=colors["muted"])
fig.tight_layout()
fig.savefig(v5_dir / "ptcg_strategy_mri_v5_resource_heuristic_matrix.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.4), gridspec_kw={"width_ratios": [1.35, 1.0]})
source_plot = external_sources.copy()
source_plot["signal_length"] = source_plot.idea_signal.str.len()
axes[0].barh(source_plot.source, source_plot.signal_length, color=[colors["green"], colors["cyan"], colors["blue"], colors["violet"]])
axes[0].set_title("External source map: four different kinds of evidence")
axes[0].set_xlabel("narrative signal length, not score")
for y, row in enumerate(source_plot.itertuples()):
    axes[0].text(row.signal_length + 3, y, row.snapshot_detail, va="center", fontsize=8, color=colors["muted"])

tier_order = {"S": 4, "A/S": 3.5, "A": 3, "B context": 2}
meta_decks["tier_score"] = meta_decks.tier.map(tier_order)
axes[1].scatter(meta_decks.tier_score, np.arange(len(meta_decks)), s=120, color=colors["gold"], edgecolor="white", linewidth=1.2)
axes[1].set_yticks(np.arange(len(meta_decks)), meta_decks.archetype, fontsize=8)
axes[1].set_xticks([2, 3, 3.5, 4], ["B ctx", "A", "A/S", "S"])
axes[1].set_title("Deck signals become roles, not copied lists")
axes[1].grid(axis="x", alpha=.25)
fig.suptitle("Strategy MRI V5: public meta as a safety layer", fontsize=16, fontweight="bold")
fig.tight_layout()
fig.savefig(v5_dir / "ptcg_strategy_mri_v5_external_source_map.png")
plt.show()

fig, ax = plt.subplots(figsize=(12.2, 5.4))
score_rows = official_rows.dropna(subset=["publicScore"]).sort_values("publicScore")
bar_colors = [colors["green"] if row == 55056992 else colors["blue"] for row in score_rows.row]
ax.barh(score_rows.row.astype(str), score_rows.publicScore, color=bar_colors)
ax.axvline(float(leaderboard_context.score.max()), color=colors["red"], lw=2, ls="--", label="observed leaderboard head")
ax.set_title("Official score boundary: exact rows only, and we are not current #1")
ax.set_xlabel("publicScore (higher is better)")
for y, row in enumerate(score_rows.itertuples()):
    ax.text(row.publicScore + 12, y, row.claim_role, va="center", fontsize=9)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(v5_dir / "ptcg_strategy_mri_v5_official_score_boundary.png")
plt.show()

fig, ax = plt.subplots(figsize=(13.0, 5.8))
run = vote_runway.sort_values("votes")
ax.barh(run.slug, run.votes, color=[colors["green"] if "strategy-mri" in slug else colors["violet"] for slug in run.slug])
ax.axvline(20, color=colors["gold"], lw=2, ls="--")
ax.set_title("PTCG vote runway: where a readable refresh can still help")
ax.set_xlabel("totalVotes observed")
ax.tick_params(axis="y", labelsize=8)
for y, row in enumerate(run.itertuples()):
    ax.text(row.votes + 0.35, y, f"{row.votes} · {row.lane}", va="center", fontsize=8.5, color=colors["muted"])
fig.tight_layout()
fig.savefig(v5_dir / "ptcg_strategy_mri_v5_vote_runway.png")
plt.show()

fig, ax = plt.subplots(figsize=(12.6, 6.0))
ax.axis("off")
ax.set_title("V5 reader card: resource conversion beats damage sorting", fontsize=16, fontweight="bold", pad=18)
cards = [
    ("Acceleration", "energy/draw engines keep the first attacker live", colors["green"]),
    ("Reach", "bench pressure converts partial damage into future turns", colors["cyan"]),
    ("Counterplay", "anti-ex and status cards punish greedy prize maps", colors["violet"]),
    ("Boundaries", "official score claims stay tied to exact COMPLETE rows", colors["red"]),
]
for i, (title, body, color) in enumerate(cards):
    x = 0.06 + i * 0.235
    ax.add_patch(plt.Rectangle((x, 0.38), 0.20, 0.32, color="white", ec=colors["line"], lw=1.4))
    ax.add_patch(plt.Rectangle((x, 0.38), 0.20, 0.035, color=color, ec=color))
    ax.text(x + 0.10, 0.60, title, ha="center", va="center", fontsize=12, fontweight="bold")
    ax.text(x + 0.10, 0.50, body, ha="center", va="center", fontsize=9.5, color=colors["muted"], wrap=True)
ax.text(0.5, 0.17, "No submission.tar.gz. No new slug. No code imported from public notebooks.", ha="center", fontsize=11.5, color=colors["red"], fontweight="bold")
fig.tight_layout()
fig.savefig(v5_dir / "ptcg_strategy_mri_v5_reader_card.png")
plt.show()

receipt = {
    "marker": PTCG_STRATEGY_MRI_EXTERNAL_META_V5_REFRESH_MARKER,
    "snapshot_utc": SNAPSHOT_UTC,
    "scope": "PTCG_STRATEGY_MRI_PUBLIC_VISUAL_EXTERNAL_META_REFRESH",
    "official_competition_submission_created": False,
    "submission_archive_created": False,
    "source_reuse_boundary": "external meta and public notebook context only; no code copied",
    "best_exact_own_complete_row": 55056992,
    "best_exact_own_complete_publicScore": 844.4,
    "latest_observed_complete_row": 55131346,
    "latest_observed_complete_publicScore": 660.8,
    "observed_public_leaderboard_head": float(leaderboard_context.score.max()),
    "current_highest_claim": False,
    "external_sources": external_sources[["source", "url"]].to_dict(orient="records"),
    "output_files": sorted(p.name for p in v5_dir.glob("ptcg_strategy_mri_v5_*")),
}
(v5_dir / "PTCG_STRATEGY_MRI_V5_EXTERNAL_META_REFRESH_RECEIPT_20260801.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
assert receipt["official_competition_submission_created"] is False
assert receipt["submission_archive_created"] is False
assert not any(Path(".").glob("submission*"))

display(external_sources)
display(heuristics)
display(official_rows)
for path in [
    v5_dir / "ptcg_strategy_mri_v5_external_source_map.png",
    v5_dir / "ptcg_strategy_mri_v5_resource_heuristic_matrix.png",
    v5_dir / "ptcg_strategy_mri_v5_official_score_boundary.png",
    v5_dir / "ptcg_strategy_mri_v5_vote_runway.png",
    v5_dir / "ptcg_strategy_mri_v5_reader_card.png",
]:
    display(Image(str(path)))
print(json.dumps(receipt, indent=2))


## V5 machine-readable receipt

This final cell exposes the exact V5 evidence boundary for readers and for
downloaded-output validation.


In [ ]:
v5_receipt = json.loads((Path("ptcg_strategy_atlas_outputs") / "PTCG_STRATEGY_MRI_V5_EXTERNAL_META_REFRESH_RECEIPT_20260801.json").read_text())
display(pd.DataFrame([{
    "marker": v5_receipt["marker"],
    "best_exact_own_row": v5_receipt["best_exact_own_complete_row"],
    "best_exact_own_score": v5_receipt["best_exact_own_complete_publicScore"],
    "observed_public_head": v5_receipt["observed_public_leaderboard_head"],
    "current_highest_claim": v5_receipt["current_highest_claim"],
    "submission_archive_created": v5_receipt["submission_archive_created"],
}]))


收到，这是该 Notebook 的完整源码。基于实际内容，我为你提炼 **prvsiyan** 的核心策略框架，以及对你参赛的直接启示：

---

## 一、核心论点（Thesis）

**"强卡组不是单卡强度排序，而是一个受约束的控制系统——其价值来自在隐藏信息（抽牌）下，能够合法触发的可达行动序列。"**

这意味着：
- 不要问"哪张卡最强"，要问"哪60张卡能在约束下稳定执行某个战术闭环"
- 单卡伤害高但能量成本高、撤退费用高、进化链长，会拖累整体一致性

---

## 二、数据层：有界解析（Bounded Parsing）

他处理卡牌数据的严谨程度值得借鉴：

| 处理项 | 原则 | 对你OCR的启示 |
|--------|------|--------------|
| **伤害** | 仅接受无条件整数（如"20"）。含"×"、"+"、条件文本的，标记为 `uncertain`，**不强行数值化** | 你的OCR结果里，遇到"20×"或"若对手已受伤则+30"这类，不能直接存为数字，要标记条件类型 |
| **能量成本** | 解析 `{F}`、`{C}`、`●` 等符号，统计无色/有色能量数量 | 确保OCR能识别能量符号，这是后续计算 DPE（伤害/能量比）的基础 |
| **缺失值** | 承认结构性缺失：能量卡无HP、训练家无撤退费用 | 数据库设计时允许NULL，不要填0或默认值 |

---

## 三、卡牌评分模型（多目标效率）

他构建了透明、可审计的评分函数：

**宝可梦效用** = 0.34×耐久Z + 0.42×压强Z + 0.18×机动Z - 0.20×条件伤害占比 - 0.10×进化深度

- **耐久**：HP
- **压强**：最佳伤害/能量比（DPE）
- **机动**：撤退费用越低越好（负向）
- **惩罚项**：条件伤害越多越不稳定；进化链越长卡手率越高

**训练家效用** = 1.0×抽卡 + 1.15×检索 + 0.65×切换 + 0.75×能量 + 0.35×回复 + 0.45×干扰

> 检索（search）权重最高，因为检索能压缩方差，提高关键卡上手率。

---

## 四、60卡构建：整数规划（MILP）

他用 `scipy.optimize.milp` 求解卡组，关键约束包括：

- **角色配额**：宝可梦18、训练家28、能量14（可调整）
- **基础宝可梦≥8张**（保证开局不卡手）
- **进化宝可梦≥4张**
- **主属性宝可梦≥12张**
- **同名卡≤4张**（普通卡）、基础能量不限
- **ACE SPEC≤1张**
- **进化链完整性**：有进化型必须有对应的基础/一阶（防止选了二阶没一阶）

**启示**：你的规则Agent如果涉及自动组卡，应该把这些作为**硬约束**，而非"尽量满足"。

---

## 五、开局一致性：精确概率

他计算起手7张中**同时有基础宝可梦+能量**的精确概率（容斥原理），并用蒙特卡洛交叉验证。

公式核心：
- P(有基础 ∧ 有能量) = 1 - P(无基础) - P(无能量) + P(无基础且无能量)

**启示**：你选胡地卡组时，可以用这个公式快速验证"8张基础+14张能量"的开局稳定性，避免凭感觉拍脑袋。

---

## 六、决策策略：层级式Policy Order

他的行动选择是**严格分层**的，不是简单打分排序：

```
legal actions → forced（强制） → immediate_win（立即获胜） → prevents_loss（止损） → value/progress（价值推进） → deterministic fallback（保底）
```

评分公式：
- `2.0×immediate_win + 1.4×prevents_loss + 0.8×progress + attack_value - 0.6×risk`

**启示**：你的规则Agent应该实现类似的**决策层级**。不要在一个大公式里混为一谈，先判断"这步是否强制/必胜/止损"，再考虑价值推进。

---

## 七、V5 Refresh：外部Meta的"安全翻译"

这是最有价值的部分——他如何把PTCG Pocket的外部Meta信息转化为Kaggle 60卡环境的启发式：

| 外部Meta概念 | 转化后的卡组构建权重 | 含义 |
|-------------|---------------------|------|
| **引擎优先一致性** (0.24) | 抽滤/检索/能量加速 | 现代卡组靠资源循环维持攻击，不是堆高伤害 |
| **储能转换** (0.18) | 能量成本、DPE、撤退 | 银行式能量（如电系充能）能爆发但怕卡手 |
| **板凳/ reach 压强** (0.17) | 效果文本中的"bench"、"anywhere" | 狙击/铺伤能转化部分伤害为未来回合优势 |
| **反ex/奖品拒绝** (0.15) | 针对ex的counter卡 | 惩罚对手过度依赖ex打手 |
| **支点与重置** (0.14) | 撤退费用、切换/回手效果 | 保留胜势局面，避免送奖品 |
| **流行度怀疑** (0.12) | 不直接复制热门卡组 | 使用率≠期望价值 |

**关键结论**：**资源转换优于伤害排序**（Resource conversion beats damage sorting）。

---

## 八、失败案例（Candidate Gates）——什么不该做

他公开了被否决的"微优化"尝试，这是最有参考价值的：

| 候选策略 | 结果 | 原因 |
|---------|------|------|
| Flutter Mane 主动技能封锁 | **HOLD** | 通过了loader测试，但**零行动分歧**（即实战中从未触发不同决策） |
| Enhanced Hammer 针对特殊能量 | **HOLD** | 16局本地测试负收益，8局退化 |
| 奖品时钟瘦身守卫 | **HOLD** | 整体负收益，多锚点退化 |
| Tarountula/Roserade 场地权重 | **HOLD** | 局部收益被Garchomp对局的精确退化抵消 |

**启示**：你的规则Agent添加新逻辑时，必须满足两个条件：
1. **必须改变实际决策**（不能是"看起来有用但从未触发"的代码）
2. **必须通过多锚点 broad screen**（不能只在特定对局有效）

---

## 九、当前成绩与定位

- **自己最佳精确行**：55056992，**844.4分**
- **公开榜头部**：1252.8（Majkel1337），差距约408分
- **当前处于**：银牌区边缘（需20票左右进银牌），金牌差距大

他明确承认**不是当前第一**，且官方分数会随时间漂移（drift），所以只引用精确到行的历史快照。

---

## 十、对你参赛的直接行动建议

### 1. 卡组构建（如果你还没定）
- 不要复制Pocket的20卡卡组，但借鉴其**角色分工**：加速/检索/狙击/撤退/反ex
- 用**约束检查表**验证你的60卡：基础≥8、能量≥12、检索链完整、撤退费用不过高
- 计算起手"基础+能量"联合概率，确保≥85%

### 2. 规则Agent决策层
```python
# 建议的决策层级伪代码
if 有强制行动: 执行
elif 有立即获胜行动: 执行
elif 有止损行动（防止下回合被拿奖）: 执行
else:
    在剩余合法行动中，按"推进价值 - 0.6×风险"打分，选最高
```

### 3. OCR校验（你之前问的）
- 伤害字段必须分三类：确定性整数 / 乘性条件（×） / 附加条件（+）
- 效果文本提取关键词：draw, search, switch, energy, heal, discard, shuffle
- 不要试图把"20×"解释成20或40，标记为 `multiplier` 并在评分时惩罚其不确定性

### 4. 每日5发的提交策略
- 不要提交"看起来很美"的微优化（如上述4个HOLD案例）
- 每次提交前问自己：**这个改动是否在本地测试中改变了至少一个决策？是否通过多对局锚点测试？**
- 优先做**能改变决策层级**的改动（如新增"止损"判断），而非调参

---

需要我基于这个框架，帮你设计一个**胡地卡组的约束检查表**或**规则Agent的决策层级伪代码**吗？